In [ ]:
import os
from sarvamai import SarvamAI
API_KEY = "sk_q1of4y0i_wYj1nhyui9wyn0Yr7OQjviax"
# Initialize Sarvam AI Client
client = SarvamAI(api_subscription_key=API_KEY)
print("Client initialized successfully!")

Client initialized successfully!


In [2]:
import base64
# Function to generate audio for a given text and language code
def generate_speech(text, lang_code, output_filename):
    response = client.text_to_speech.convert(
        model="bulbul:v3",
        text=text,
        target_language_code=lang_code,
        speaker="shubh",
        pace=1.0,
    )

    audios = getattr(response, "audios", None)
    if isinstance(audios, list) and audios:
        audio_payload = audios[0]
    elif audios:
        audio_payload = audios
    else:
        raise ValueError("No audio payload returned by the API response.")

    audio_bytes = base64.b64decode(audio_payload)

    with open(output_filename, "wb") as f:
        f.write(audio_bytes)

    print(f"Saved audio to: {output_filename}")

# Testing for 4 languages
test_data = [
    (" नमस्ते, आप कैसे हैं?", "hi-IN", "hindi.mp3"),      # Hindi
    (" नमస్కారం, మీరు ఎలా ఉన్నారు?", "te-IN", "telugu.mp3"),  # Telugu
    (" வணக்கம், நீங்கள் எப்படி இருக்கிறீர்கள்?", "ta-IN", "tamil.mp3"),  # Tamil
    (" Hello, how can i help you today?", "en-IN", "english.mp3") # English
]

for text, lang, filename in test_data:
    print(f"Generating speech for {lang}...")
    generate_speech(text, lang, filename)

Generating speech for hi-IN...
Saved audio to: hindi.mp3
Generating speech for te-IN...
Saved audio to: telugu.mp3
Generating speech for ta-IN...
Saved audio to: tamil.mp3
Generating speech for en-IN...
Saved audio to: english.mp3


In [3]:
# Function to transcribe audio files
def transcribe_audio(audio_filename, mode="transcribe"):
    try:
        with open(audio_filename, "rb") as audio_file:
            response = client.speech_to_text.transcribe(
                file=audio_file,
                model="saaras:v3",
                mode=mode,  # "transcribe" keeps original language, "translate" translates to English
            )
        
        print(f"--- Result for {audio_filename} ---")
        print(f"Transcript: {response.transcript}\n")
        return response.transcript

    except Exception as e:
        print(f"Error transcribing {audio_filename}: {e}")

# Transcribe the audio files generated in Step 2
print("--- Starting Speech-to-Text ---\n")
transcribe_audio("hindi.mp3")
transcribe_audio("telugu.mp3")
transcribe_audio("tamil.mp3")
transcribe_audio("english.mp3")

--- Starting Speech-to-Text ---

--- Result for hindi.mp3 ---
Transcript: नमस्ते, आप कैसे हैं?

--- Result for telugu.mp3 ---
Transcript: నమస్కారం, మీరు ఎలా ఉన్నారు?

--- Result for tamil.mp3 ---
Transcript: வணக்கம், நீங்கள் எப்படி இருக்கிறீர்கள்?

--- Result for english.mp3 ---
Transcript: Hello, how can I help you today?



'Hello, how can I help you today?'

In [4]:
# Translates foreign speech directly into English
transcribe_audio("hindi.mp3", mode="translate")

--- Result for hindi.mp3 ---
Transcript: Hello, how are you?



'Hello, how are you?'

In [ ]:
import base64
import ipywidgets as widgets
from IPython.display import Audio, display
from sarvamai import SarvamAI

API_KEY = "sk_q1of4y0i_wYj1nhyui9wyn0Yr7OQjviax"
client = SarvamAI(api_subscription_key=API_KEY)

LANGUAGE_CODES = {
    "English": "en-IN",
    "Hindi": "hi-IN",
    "Telugu": "te-IN",
    "Tamil": "ta-IN",
}

output_area = widgets.Output()

input_lang = widgets.Dropdown(
    options=list(LANGUAGE_CODES.keys()),
    value="English",
    description="Input language:"
)

output_lang = widgets.Dropdown(
    options=list(LANGUAGE_CODES.keys()),
    value="Hindi",
    description="Output language:"
)

text_input = widgets.Textarea(
    value="Hello, how are you today?",
    placeholder="Enter text to convert",
    description="Text:",
    layout=widgets.Layout(width="85%")
)

run_button = widgets.Button(description="Run")


def translate_text(text, source_lang_code, target_lang_code):
    if source_lang_code == target_lang_code:
        return text

    response = client.text.translate(
        input=text,
        source_language_code=source_lang_code,
        target_language_code=target_lang_code,
        speaker_gender="Male",
        model="sarvam-translate:v1",
        mode="formal",
    )
    return response.translated_text


def generate_speech(text, lang_code, output_filename):
    response = client.text_to_speech.convert(
        model="bulbul:v3",
        text=text,
        target_language_code=lang_code,
        speaker="shubh",
        pace=1.0,
    )

    audios = getattr(response, "audios", None)
    if isinstance(audios, list) and audios:
        audio_payload = audios[0]
    elif audios:
        audio_payload = audios
    else:
        raise ValueError("No audio payload returned by the API response.")

    audio_bytes = base64.b64decode(audio_payload)
    with open(output_filename, "wb") as f:
        f.write(audio_bytes)

    print(f"Saved audio to: {output_filename}")


def run_selected_mode(_=None):
    with output_area:
        output_area.clear_output(wait=True)
        source_code = LANGUAGE_CODES[input_lang.value]
        target_code = LANGUAGE_CODES[output_lang.value]
        source_text = text_input.value.strip()

        if not source_text:
            print("Please enter text to convert.")
            return

        print(f"Translating and generating speech from {input_lang.value} to {output_lang.value}...")
        translated_text = translate_text(source_text, source_code, target_code)
        generate_speech(translated_text, target_code, "generated_speech.mp3")
        display(Audio(filename="generated_speech.mp3"))
        print("Translated text:")
        print(translated_text)


run_button.on_click(run_selected_mode)
display(input_lang, output_lang, text_input, run_button, output_area)

Dropdown(description='Input language:', options=('English', 'Hindi', 'Telugu', 'Tamil'), value='English')

Dropdown(description='Output language:', index=1, options=('English', 'Hindi', 'Telugu', 'Tamil'), value='Hind…

Textarea(value='Hello, how are you today?', description='Text:', layout=Layout(width='85%'), placeholder='Ente…

Button(description='Run', style=ButtonStyle())

Output()

In [ ]:
import base64
import ipywidgets as widgets
from IPython.display import Audio, display
from sarvamai import SarvamAI

API_KEY = "sk_q1of4y0i_wYj1nhyui9wyn0Yr7OQjviax"
client = SarvamAI(api_subscription_key=API_KEY)

LANGUAGE_CODES = {
    "English": "en-IN",
    "Hindi": "hi-IN",
    "Telugu": "te-IN",
    "Tamil": "ta-IN",
}

output_area = widgets.Output()

input_lang = widgets.Dropdown(
    options=list(LANGUAGE_CODES.keys()),
    value="English",
    description="Input language:"
)

output_lang = widgets.Dropdown(
    options=list(LANGUAGE_CODES.keys()),
    value="Hindi",
    description="Output language:"
)

file_upload = widgets.FileUpload(
    accept=".wav,.mp3,.m4a,.ogg",
    multiple=False,
    description="Upload audio"
)

run_button = widgets.Button(description="Run STT")


def translate_text(text, source_lang_code, target_lang_code):
    if source_lang_code == target_lang_code:
        return text

    response = client.text.translate(
        input=text,
        source_language_code=source_lang_code,
        target_language_code=target_lang_code,
        speaker_gender="Male",
        model="sarvam-translate:v1",
        mode="formal",
    )
    return response.translated_text


def save_uploaded_audio():
    uploaded_files = file_upload.value or []
    if not uploaded_files:
        raise ValueError("Please upload an audio file first.")

    uploaded_file = uploaded_files[0]
    uploaded_name = uploaded_file.get("name", "uploaded_audio.wav")
    content = uploaded_file.get("content")

    if content is None:
        raise ValueError("No file content was captured from the upload.")

    if hasattr(content, "tobytes"):
        content_bytes = content.tobytes()
    else:
        content_bytes = bytes(content)

    ext = "." + uploaded_name.split(".")[-1].lower() if "." in uploaded_name else ".wav"
    audio_path = f"uploaded_audio{ext}"

    with open(audio_path, "wb") as f:
        f.write(content_bytes)

    return audio_path


def transcribe_audio(audio_path, source_lang_code):
    with open(audio_path, "rb") as audio_file:
        response = client.speech_to_text.transcribe(
            file=audio_file,
            model="saaras:v3",
            mode="transcribe",
        )

    transcript = getattr(response, "transcript", None)
    if not transcript or not str(transcript).strip():
        raise ValueError("No transcript was returned for the uploaded audio.")

    return transcript


def run_selected_mode(_=None):
    with output_area:
        output_area.clear_output(wait=True)
        source_code = LANGUAGE_CODES[input_lang.value]
        target_code = LANGUAGE_CODES[output_lang.value]

        try:
            audio_path = save_uploaded_audio()
            print(f"Using uploaded audio: {audio_path}")
            display(Audio(filename=audio_path))

            transcript = transcribe_audio(audio_path, source_code)
            translated_text = translate_text(transcript, source_code, target_code)

            print("Transcript:")
            print(transcript)
            print("Translated text:")
            print(translated_text)
        except Exception as exc:
            print(f"STT failed: {exc}")


run_button.on_click(run_selected_mode)
display(input_lang, output_lang, file_upload, run_button, output_area)

Dropdown(description='Input language:', options=('English', 'Hindi', 'Telugu', 'Tamil'), value='English')

Dropdown(description='Output language:', index=1, options=('English', 'Hindi', 'Telugu', 'Tamil'), value='Hind…

FileUpload(value=(), accept='.wav,.mp3,.m4a,.ogg', description='Upload audio')

Button(description='Run STT', style=ButtonStyle())

Output()